# Lab 15: The Bias-Variance Tradeoff — Watching a Model Overfit
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 65 min Core + 30 min Write it yourself + 25 min Extension + take-home Challenge

---

**Learning Objectives:**
- Decompose prediction error into bias, variance and irreducible noise
- Fit models across a complexity range and locate where held-out error turns
- Show that the training error curve cannot tell you where to stop
- Select a model by cross-validation rather than a single split
- Write a small function that gives the k-fold cross-validation error for any polynomial degree and number of folds

**Dataset:** Simulated nonlinear data with a known truth

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-4 are GUIDED: run each cell as it is, then answer the questions. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**Plan on about 2 hours** (1 hour 55 minutes) for the required work: setup, Parts 1-4, Write it yourself, and the README and submission.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Write it yourself (required),** after Part 4: `kfold_mse()`, a function that gives back the k-fold cross-validation error of a polynomial of a given degree. About seven lines from a blank cell. The check cell below it must print "Passed"; it then uses your function to print the 10-fold errors for degrees 4 and 5.
> - The answers to the interpretation questions after each part, in text cells: one for Part 1, two each for Parts 2 and 3, and one for Part 4.
>
> **You run and read:** everything else. Setup and Parts 1-4, including Part 4's nested cross-validation, are working code to learn from. Read each cell before you run it, and make sure you can say what each line does.
>
> **The Extension** (optional) and **the Challenge** (take-home) have no starter code: you write them in new cells, from the task. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Every random draw in this lab is seeded where it happens, by `sample()` in Step 2 or by
`random_state=RANDOM_STATE` in a fold split. So running a cell twice gives the same numbers.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the seed
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold      # new in this lab, used from Part 3 on
from pathlib import Path

RANDOM_STATE = 42     # the seed for the 60 training points (Step 2) and for every fold split

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: Two Ways to Be Wrong (GUIDED — 20 min)

A model can miss because it is too rigid to follow the signal, or because it is
flexible enough to follow the noise. In a single fit the two can look alike.

This lab works in a simulated world, so the truth is known: y is the curve sin(1.4x) + 0.25x plus
noise with a standard deviation of 0.45, and x runs from 0 to 6. Real data never shows you the
curve; here every fit can be checked against it.

Step 2 writes that world as two functions. `truth(x)` is the curve. `sample(n, seed)` draws n
points: x spread evenly at random between 0 and 6, and y the curve plus noise. Each call builds its
own random generator from `seed`, so the same seed always gives the same points, and a new seed a
fresh sample from the same world.

In [ ]:
# GUIDED — run as-is
# Step 2: a curved truth, observed with noise
def truth(x):
    """The true curve, which a real analyst never sees."""
    return np.sin(1.4 * x) + 0.25 * x


def sample(n, seed):
    """n points from this world: x between 0 and 6, y the true curve plus noise with sd 0.45."""
    r = np.random.default_rng(seed)
    x = r.uniform(0, 6, n)
    return x, truth(x) + r.normal(0, 0.45, n)


x_tr, y_tr = sample(60, RANDOM_STATE)     # the one training sample that Parts 1-4 fit

Step 3 fits polynomials of degree 1, 4 and 18 to those 60 points and draws each one beside the
truth. `np.polyfit(x, y, deg)` fits a polynomial of degree `deg` by least squares (Lab 5 used it
with degree 1, a straight line) and gives back its coefficients, highest power first.
`np.polyval(coefs, x)` evaluates that polynomial at each x. `np.linspace(0, 6, 300)` makes 300
evenly spaced x values to draw smooth curves through, and `zip` pairs each panel with one degree,
as in Lab 10.

In [ ]:
# GUIDED — run as-is
# Step 3: three degrees, fitted to the same 60 points
grid = np.linspace(0, 6, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, deg in zip(axes, [1, 4, 18]):
    c = np.polyfit(x_tr, y_tr, deg)
    ax.scatter(x_tr, y_tr, s=14, alpha=0.6, color="darkslategray")
    ax.plot(grid, truth(grid), color="seagreen", lw=2, label="truth")
    ax.plot(grid, np.polyval(c, grid), color="firebrick", lw=2, label=f"degree {deg}")
    ax.set_ylim(-3, 4)
    ax.set_title(f"degree {deg}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("figures/ch15_underfit_overfit.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** on the left, a line too rigid to follow the curve. On the right, a curve that
bends to chase individual points. The middle one tracks the truth closely, but you can see that only
because the green truth is drawn. From the 60 points alone, you could not tell.

Step 4 measures the two kinds of error directly, which only a simulation can do, because it can
draw as many samples as it likes. For each degree it draws 200 fresh samples of 60 points (seeds
5000 to 5199), fits each one, and records its predictions at 40 fixed x values, `x_eval`. That
fills a table `preds` with 200 rows, one per sample, and 40 columns, one per x value.

- **bias^2** averages the 200 predictions at each x, compares that average with the truth, squares
  the gap, and averages over the 40 x values. How far is the fit from the truth, on average over
  samples?
- **variance** is the variance of the 200 predictions at each x, averaged over the 40 x values. How
  much does the fit move from one sample to the next?
- **bias^2+var** is their sum: the part of the error that the choice of model can change. The
  error on a new point adds the noise itself, 0.45² = 0.2025, which no model can remove.

`axis=0` means "down the rows": `preds.mean(axis=0)` gives one mean per column, that is, per x
value. `np.empty((REPS, len(x_eval)))` makes a table of that shape to fill, and `preds[r_i] = ...`
fills row `r_i`. `rows.append((deg, bias2, var, bias2 + var))` stores four numbers together as one
row, and `for d, b, v, t in rows` unpacks each row back into four names.

In [ ]:
# GUIDED — run as-is
# Step 4: bias squared and variance, from 200 fresh samples at each degree
x_eval = np.linspace(0.5, 5.5, 40)      # 40 x values, kept back from the ends of 0 to 6
REPS = 200
rows = []
for deg in [1, 2, 3, 5, 9, 15]:
    preds = np.empty((REPS, len(x_eval)))
    for r_i in range(REPS):
        xs, ys = sample(60, 5000 + r_i)            # a fresh sample, the same size as x_tr
        preds[r_i] = np.polyval(np.polyfit(xs, ys, deg), x_eval)
    bias2 = np.mean((preds.mean(axis=0) - truth(x_eval)) ** 2)
    var = np.mean(preds.var(axis=0))
    rows.append((deg, bias2, var, bias2 + var))

# >8 right-aligns in 8 characters; >11.4f right-aligns a number in 11, with 4 decimals
print(f"{'degree':>8}{'bias^2':>11}{'variance':>11}{'bias^2+var':>13}")
print("-" * 43)       # a string times a number repeats it: a rule of 43 dashes
for d, b, v, t in rows:
    print(f"{d:>8}{b:>11.4f}{v:>11.4f}{t:>13.4f}")

**Expected output:** six rows, for degrees 1, 2, 3, 5, 9 and 15. bias^2 reads 0.5001, 0.2031,
0.1237, 0.0036, 0.0002 and 0.0003. variance reads 0.0211, 0.0187, 0.0205, 0.0180, 0.0321 and
0.0994. bias^2+var is smallest at degree 5, at 0.0216.

**The table against the textbook drawing.** The drawing has bias falling and variance rising across
the whole range. In this table, bias^2 falls by more than three orders of magnitude and then stops,
near zero, from degree 9 on. Variance is roughly flat, about 0.02, from degree 1 to degree 5, and
only then climbs, to about five times its degree-5 value at degree 15. When you report this, report
what the table shows, not the drawing.

### Interpretation Questions

1. Bias squared falls from 0.50 at degree 1 to about 0.0002 by degree 9, then stops. Explain the fall in terms of what the polynomial family can represent, and explain why it stops near zero rather than falling further.

*Your answer here:*

## Part 2: The Curve That Misleads (GUIDED — 15 min)

Step 5 fits degrees 1 to 15, each once, to the 60 training points, and scores every fit twice: on
those same 60 points (the training error) and on 2,000 new points from the same world (the held-out
error). A real analyst would not have 2,000 spare points; the simulation lends them to us, so that
we can see the true shape of the curve. The error is the mean squared error, the MSE: Lab 12's RMSE
before the square root.

`2_000` is 2000: Python ignores the underscore, which is there for the eye. `np.argmin(te_err)`
gives the *position* of the smallest held-out error in the list. Positions count from 0 and the
degrees from 1, so the degree is the position plus 1; `int(...)` turns NumPy's whole number into a
plain Python one, which prints without decoration. `list(degrees)` spells the range out as a list of
the 15 degrees, for the x-axis. In `ax.plot`, `"o-"` draws dots joined by a
line, and `"s-"` squares. `set_yscale("log")` puts the y-axis on a log scale, so that small and
large errors are both readable.

In [ ]:
# GUIDED — run as-is
# Step 5: training error against held-out error, degrees 1 to 15
x_te, y_te = sample(2_000, 99)      # 2,000 new points from the same world, with another seed
degrees = range(1, 16)
tr_err, te_err = [], []             # two empty lists
for deg in degrees:
    c = np.polyfit(x_tr, y_tr, deg)                              # fitted on the 60 only
    tr_err.append(np.mean((np.polyval(c, x_tr) - y_tr) ** 2))    # scored on the same 60
    te_err.append(np.mean((np.polyval(c, x_te) - y_te) ** 2))    # scored on the 2,000

best = int(np.argmin(te_err)) + 1   # position 0 holds degree 1

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(degrees), tr_err, "o-", label="training error", color="darkslategray")
ax.plot(list(degrees), te_err, "s-", label="held-out error", color="firebrick")
ax.axvline(best, ls="--", color="gray")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("mean squared error")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.savefig("figures/ch15_error_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Held-out error is minimised at degree {best}.")

**Expected output:** "Held-out error is minimised at degree 5." In the chart, the training error
falls at every step from degree 1 to 15. The held-out error falls to the dashed line at degree 5,
then rises, unevenly.

### Interpretation Questions

1. The training curve is monotone. Say why, and connect it to what you saw about R-squared in Lab 12.
2. Name the two halves of the held-out curve in bias-variance terms.

*Your answers here:*

1. 
2. 

## Part 3: Cross-Validation (GUIDED — 18 min)

Without 2,000 spare points, cross-validation does the scoring with the 60 you have. 5-fold CV deals
the 60 points into five folds of 12. Each fold takes one turn as the test points while the
polynomial is fitted on the other 48, and the five test errors are averaged.

`KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` sets up that deal: shuffle the points
first, with a fixed seed, so everyone gets the same folds. `kf.split(x_tr)` then gives five pairs of
position arrays, one pair per fold: the positions of the 48 training points and of the 12 test
points. `x_tr[tr_i]` picks out the points at those positions. `np.std(errs)` is the spread of the
five fold errors; NumPy's `std` divides by n, not n − 1, unless you pass `ddof=1`. In the last
print, `\n` starts a new line.

In [ ]:
# GUIDED — run as-is
# Step 6: 5-fold cross-validation for degrees 1 to 11, with the spread across folds
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"{'degree':>8}{'CV MSE':>11}{'std across folds':>19}")
print("-" * 40)
cv_means = []
for deg in range(1, 12):
    errs = []                                     # this degree's five test-fold errors
    for tr_i, te_i in kf.split(x_tr):
        c = np.polyfit(x_tr[tr_i], y_tr[tr_i], deg)                           # fit on 48
        errs.append(np.mean((np.polyval(c, x_tr[te_i]) - y_tr[te_i]) ** 2))   # score the other 12
    cv_means.append(np.mean(errs))
    print(f"{deg:>8}{np.mean(errs):>11.4f}{np.std(errs):>19.4f}")

print(f"\nCV picks degree {int(np.argmin(cv_means)) + 1}; the held-out set picked {best}.")

**Expected output:** eleven rows. The CV MSE is lowest at degree 4, 0.1401, with degree 5 next at
0.1510, and the last line reads "CV picks degree 4; the held-out set picked 5." The spread across
folds is 0.0223 at degree 4 and 0.0267 at degree 5, and between 0.10 and 0.18 at every other degree.

0.1401 is below the noise variance of 0.2025 only because these 60 points happen to be quiet (their
own noise variance is 0.122); on Step 5's 2,000 points, degree 4's error is 0.2284.

The class poll's standard error of 0.0112 for degree 4 is this table's spread recomputed with
`ddof=1` and divided by √5, the square root of the number of folds.

### Interpretation Questions

1. Step 5's 2,000 held-out points are a luxury of simulation. With only the 60 training points, would you choose the degree with one split, 48 points to fit and 12 to score, or with 5-fold CV? Why?
2. The spread across folds is 0.02-0.03 at degrees 4 and 5, and 0.10 or more at every other degree. What is that telling you beyond the mean?

*Your answers here:*

1. 
2. 

## Part 4: The Selection You Did Not Account For (GUIDED — 12 min)

In Step 6 you chose degree 4 because its CV error, 0.1401, was the smallest of eleven. That 0.1401
is the number you would naturally report as the model's expected error on new data. Is it a fair
estimate?

Nested cross-validation separates the two jobs, choosing and scoring:

- the **outer** loop holds out a fold that is used for nothing except the final score. No choice is
  ever made with it;
- the **inner** loop, inside each outer training set of 48 points, runs the same degree search as
  Step 6 and picks a winner.

The outer score then measures the whole procedure, search included, rather than the one degree the
search happened to find.

Step 7 is written for you. Read it against Step 6: the inner loop is Step 6's loop, run on `x_in`
and `y_in` in place of `x_tr` and `y_tr`. `DEGREES` is the range 1 to 11, and `list(DEGREES)` turns
it into a list, so `list(DEGREES)[i]` is the degree at position i.

In [ ]:
# GUIDED — run as-is
# Step 7: nested cross-validation. Is the smallest CV error a fair estimate?
DEGREES = range(1, 12)

outer = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
outer_scores, chosen_degrees = [], []

for out_tr, out_te in outer.split(x_tr):
    x_in,  y_in  = x_tr[out_tr], y_tr[out_tr]   # choose the degree with these 48 points
    x_out, y_out = x_tr[out_te], y_tr[out_te]   # score on these 12, and use them for nothing else

    # Inner loop: Step 6's degree search, on x_in and y_in only
    inner = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    inner_means = []
    for deg in DEGREES:
        fold_errs = []
        for in_tr, in_te in inner.split(x_in):
            c = np.polyfit(x_in[in_tr], y_in[in_tr], deg)
            fold_errs.append(np.mean((np.polyval(c, x_in[in_te]) - y_in[in_te]) ** 2))
        inner_means.append(np.mean(fold_errs))
    best_deg = list(DEGREES)[int(np.argmin(inner_means))]   # this outer fold's winner
    chosen_degrees.append(best_deg)

    # Outer score: refit the winner on all 48 inner points, then score the 12 nothing above touched
    c = np.polyfit(x_in, y_in, best_deg)
    outer_scores.append(np.mean((np.polyval(c, x_out) - y_out) ** 2))

nested = np.mean(outer_scores)          # averaged over the five outer folds, as Step 6 averages

print(f"Smallest CV MSE, Step 6 : {min(cv_means):.4f}")
print(f"Nested CV estimate      : {nested:.4f}")
print(f"degree chosen in each outer fold: {chosen_degrees}")

**Expected output:** the smallest CV MSE, 0.1401, a nested CV estimate of 0.2070, and the degree
chosen in each outer fold, [8, 8, 4, 4, 6].

### Interpretation Questions

1. Which is larger here, Step 6's smallest CV MSE or the nested CV estimate? On a different sample from this world, would you expect the same order every time, or on average? Say why.

*Your answer here:*

---

## Write it yourself (required — 30 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Step 6 used 5 folds. Would 10 folds rank degrees 4 and 5 the same way? Rather than
copy Step 6's loop a third time, turn it into a function.

Write a function `kfold_mse(x, y, deg, k)` that deals the points into `k` folds the way Step 6 does
(shuffled, with `random_state=RANDOM_STATE`), fits a polynomial of degree `deg` on each fold's
training points, scores it on that fold's test points, and gives back the average of the `k` test
errors, as one number. With Step 6's settings, `kfold_mse(x_tr, y_tr, 4, 5)` gives Step 6's 0.1401.
Once the check passes, it uses your function to answer the question: it prints the 10-fold errors
for degrees 4 and 5 beside Step 6's 5-fold ones.

- It takes about seven lines: the `def` line, the `KFold` line, an empty list, the `for` line, the
  fit, the append and the `return`.
- Step 6 has the body you need: the lines inside `for deg in range(1, 12):`. Inside the function,
  use the four arguments wherever Step 6 names its points, its degree and its number of folds.
- Give back the mean squared error averaged over the folds: not the fitted coefficients, not the
  list of fold errors, and not a square root.
- Lab 12's `holdout_rmse()` shows the shape of a function: the `def` line, the body, and `return`.

Then run the check cell. It calls your function with Step 6's settings and compares the answers with
Step 6's table; it also tries it on changed copies of the data and with other numbers of folds.
`assert` stops with its message when its condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): kfold_mse(x, y, deg, k)

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "kfold_mse" not in globals():
    print("Not written yet: write kfold_mse() in the cell above and run it, then run this check.")
else:
    def call(x, y, deg, k):
        """Run your kfold_mse; if it stops with an error, say where to look."""
        # try/except catches an error inside your function, so the message can name the cause
        try:
            return kfold_mse(x, y, deg, k)
        except Exception as e:
            hint = "Read the error above: it says what went wrong."
            if "positional argument" in str(e) or "n_splits" in str(e) or "Integral" in str(e):
                hint = "kfold_mse takes four arguments, in this order: x, y, deg, k."
            elif isinstance(e, IndexError) and len(x) < len(x_tr):
                hint = ("It was handed fewer points than x_tr holds, and reached past them. Does it "
                        "split its x argument, or the notebook's x_tr?")
            # str(e)[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"kfold_mse on {len(x)} points, degree {deg}, {k} folds, stopped "
                                 f"with an error:\n  {str(e)[:200]}\n{hint}") from None

    yours = call(x_tr, y_tr, 4, 5)
    assert yours is not None, "kfold_mse gave back nothing. Does it end with a return line?"
    if np.size(yours) != 1:
        # degree 2 has 3 coefficients, and 5 folds give 5 fold errors
        n_back = np.size(call(x_tr, y_tr, 2, 5))
        assert n_back != 3, ("kfold_mse gives back a polynomial's coefficients, the fitted model. "
                             "Give back the average test error.")
        assert n_back != 5, "kfold_mse gives back the five fold errors. Give back their average."
    # np.ndim is 0 for a single number
    assert np.ndim(yours) == 0 and isinstance(yours, (int, float, np.floating)), (
        "kfold_mse gives back something other than one number. Give back the average test error alone.")
    if not np.isclose(yours, cv_means[3]):
        # with deg and k the wrong way round, asking for (5, 4) gets 5 folds of degree 4
        assert not np.isclose(call(x_tr, y_tr, 5, 4), cv_means[3]), (
            "deg and k are the wrong way round. kfold_mse takes four arguments, in this order: "
            "x, y, deg, k.")
    assert np.isclose(call(x_tr, y_tr, 4, 5), yours), (
        "Two calls with the same arguments give different answers, so the folds change every time. "
        "Fix them with random_state=RANDOM_STATE, as Step 6 does.")

    # Step 5's tr_err[3] is degree 4's error on all 60 points, the points it was fitted to
    assert not np.isclose(yours, tr_err[3]), (
        "Your answer is degree 4's training error from Step 5: the polynomial was fitted on all 60 "
        "points, test folds included. Fit it on each fold's training points only.")
    assert not yours < tr_err[3], (
        "Your answer is below degree 4's training error, so each fit was scored on the points it "
        "was fitted to. Score it on the fold's test points.")

    # Doubling every y doubles every error, so a mean of squared errors grows 4 times
    ratio = call(x_tr, 2 * y_tr, 4, 5) / yours
    assert not np.isclose(ratio, 1), (
        "Doubling the y you pass in leaves your answer unchanged. Does the function use its y "
        "argument, or the notebook's y_tr?")
    assert not np.isclose(ratio, 2), (
        "Doubling y doubles your answer, so it is in the units of y, like a root mean squared "
        "error or a mean absolute error. Give back the mean of the squared errors, with no square root.")
    # x_tr[::-1] is x_tr in reverse order
    assert not np.isclose(call(x_tr[::-1], y_tr, 4, 5), yours), (
        "Pairing the y values with the x values in reverse order leaves your answer unchanged. "
        "Does the function use its x argument, or the notebook's x_tr?")
    assert not np.isclose(call(x_tr[::-1], y_tr[::-1], 4, 5), yours), (
        "Reversing the order of the points leaves your answer unchanged, so your folds are blocks "
        "of points taken in order. Shuffle them, as Step 6 does.")
    # x_tr[:48] is the first 48 points: a function that splits x_tr itself reaches past them
    assert not np.isclose(call(x_tr[:48], y_tr[:48], 4, 5), yours), (
        "kfold_mse gives the same answer for the first 48 points as for all 60. Does it use its x "
        "and y arguments everywhere, the split included?")
    assert not np.isclose(call(x_tr, y_tr, 1, 5), yours), (
        "kfold_mse gives the same answer for degree 1 as for degree 4. Does it use its deg argument?")
    assert not np.isclose(call(x_tr, y_tr, 4, 10), yours), (
        "kfold_mse gives the same answer with 10 folds as with 5. Does it use its k argument, or "
        "Step 6's kf?")
    assert not np.isclose(yours, 12 * cv_means[3]), (
        "Your answer is 12 times Step 6's: each fold's error is a sum of squared errors over its "
        "12 points. Take the mean.")
    assert not np.isclose(yours, 5 * cv_means[3]), (
        "Your answer is 5 times Step 6's: you added up the five fold errors. Average them.")
    # round(v, places) keeps that many decimals; try 2 to 5 of them
    for places in [2, 3, 4, 5]:
        assert not (np.isclose(yours, round(cv_means[3], places), rtol=0, atol=1e-9)
                    and not np.isclose(yours, cv_means[3], rtol=0, atol=1e-9)), (
            "Your number is rounded. Give back the full number; round only when you print.")

    # Step 6's errs still holds the five fold errors of degree 11, the last degree its loop ran;
    # errs[0] is the first and errs[-1] the last
    eleven = call(x_tr, y_tr, 11, 5)
    assert not np.isclose(eleven, errs[0]), (
        "For degree 11, kfold_mse gives the first fold's error alone. Is the return line inside "
        "the for loop? Move it out, so that all k folds run first.")
    assert not np.isclose(eleven, errs[-1]), (
        "For degree 11, kfold_mse gives the last fold's error alone: each pass through the loop "
        "throws away the folds before it. Make the empty list before the loop, and append to it.")
    for d in range(1, 12):
        mine = call(x_tr, y_tr, d, 5)
        assert np.isclose(mine, cv_means[d - 1]), (
            f"kfold_mse(x_tr, y_tr, {d}, 5) gives {mine:.4f}, and Step 6 printed "
            f"{cv_means[d - 1]:.4f} for degree {d}. Check three things against Step 6: the folds "
            "(shuffle=True and random_state=RANDOM_STATE), the order of np.polyfit's arguments "
            "(x, then y), and how the fold errors are combined (their mean).")

    print("Passed. kfold_mse matches every row of Step 6, and uses the data, degree and folds it is given.")
    print(f"10-fold CV MSE, from your function: degree 4 {call(x_tr, y_tr, 4, 10):.4f}, "
          f"degree 5 {call(x_tr, y_tr, 5, 10):.4f}   (5-fold, Step 6: {cv_means[3]:.4f} and "
          f"{cv_means[4]:.4f})")

---

## Extension (Optional — 25 min): More Data Instead of Less Complexity

There is no starter code: write it in the cell below.

Keep the degree fixed (degree 9, say) and let the training-set size n run from 20 to 2,000. For each
n, draw a training sample with `sample(n, seed)` (a different seed for each n), fit the polynomial,
and record its training error and its error on Step 5's 2,000 held-out points, `x_te` and `y_te`.
Plot both errors against n; a log scale on the x-axis, `ax.set_xscale("log")`, helps.

**Your answer.** The two curves come together as n grows. Explain what the gap between them
measures and what the level they settle at measures, and say which of the two a bigger dataset can
fix.

In [ ]:
# EXTENSION (optional) — write your learning curves here

*Your answer here:*

---

## Challenge (Take-Home): Where the Tradeoff Is Not a Curve

There is no starter code: write your answer in the cell below and in new cells under it.

1. Fit a polynomial of degree 40 or more to the 60 training points with `np.polyfit`, and report
   its error on Step 5's 2,000 held-out points. `np.polyfit` will warn that the fit "may be poorly
   conditioned". So far, this behaves the way Part 2 predicts.
2. Now go well past 60 coefficients. With at least as many coefficients as points, a polynomial can
   pass through every training point, and least squares has many solutions. Take the one with the smallest coefficients, the minimum-norm fit.
   Rescale x to run from −1 to 1 (`x / 3 - 1`). `np.polynomial.legendre.legvander(x, deg)` builds a
   table with one column per term, from degree 0 to `deg`, using Legendre polynomials, which stay
   well-behaved on −1 to 1 where plain powers of x do not. `np.linalg.pinv(V) @ y` gives the
   coefficients: the least-squares fit when there is one, and the one with the smallest
   coefficients when many fit equally well. Step the degree with `range(1, 1001, 10)`, record the
   training and held-out errors, and plot both against degree on a log y-axis
   (`ax.set_yscale("log")`): they span more than 25 orders of magnitude. It may not behave the way
   Part 2 predicts.
3. Read about double descent and say, in your own words, what the classical U-shaped curve assumes
   that this violates.
4. State what you would still tell an undergraduate to do in practice.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* In a simulated world with a known true curve, split the error of
  polynomial fits into bias squared and variance, using 200 fresh
  samples at each of six degrees
* Compared training error with error on 2,000 new points for degrees
  1 to 15, and found the degree where the held-out error was lowest
* Chose a degree by 5-fold cross-validation, and looked at how much
  the error varied from fold to fold
* Used nested cross-validation to check whether the smallest CV error
  is a fair estimate of the error on new data
* Wrote a small function, kfold_mse(), that gives the k-fold CV error
  for any degree and number of folds, and checked it with 10 folds
* Key findings: variance [YOUR VALUE] at degree 5 and [YOUR VALUE] at
  degree 15; lowest held-out error at degree [YOUR VALUE]; 5-fold CV
  chose degree [YOUR VALUE] with CV MSE [YOUR VALUE]; nested CV
  estimate [YOUR VALUE]; 10-fold CV MSE [YOUR VALUE] for degree 4 and
  [YOUR VALUE] for degree 5
* (Only if I did the take-home Challenge) Held-out error past the
  interpolation point: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: The Bias-Variance Tradeoff: Choosing How Flexible a Model Should Be
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab15-bias-variance`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab15-bias-variance`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab15-bias-variance`, branch `main`,
   commit message `Lab 15 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.